# Phase 5 — Generation (greedy & beam search)

A trained model gives a distribution over the next token; **decoding** turns that into
an actual sentence. We build two strategies and feel the difference:

- **Greedy** — take the argmax at each step. Fast, simple, myopic.
- **Beam search** — keep the `k` best partial hypotheses, length-normalize the scores
  (beam=4, α=0.6, as in the paper §6). Slower, usually more fluent/faithful.

Then we **visualize cross-attention during generation** — does each generated German
token look at the right English source words?

We load the checkpoint saved by `04_training.ipynb`; if it isn't there, we train a
quick model so this notebook is self-contained.

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import pathlib
import torch
import matplotlib.pyplot as plt
from tokenizers import Tokenizer

from transformer.config import ModelConfig, TrainConfig
from transformer.tokenizer import train_joint_bpe, PAD_ID, BOS_ID, EOS_ID
from transformer.data import load_multi30k, make_dataloader, collate_fn, TranslationDataset
from transformer.transformer import Transformer
from transformer.train import fit
from transformer.generate import greedy_decode, beam_search

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device()
torch.manual_seed(0)
print("device:", device)

data, _ = load_multi30k()

In [ ]:
CKPT = pathlib.Path("../checkpoints/nb_phase4.pt")
if CKPT.exists():
    blob = torch.load(CKPT, map_location=device, weights_only=False)
    tokenizer = Tokenizer.from_str(blob["tokenizer"])
    cfg = ModelConfig(**blob["model_config"])
    model = Transformer(cfg).to(device)
    model.load_state_dict(blob["model_state_dict"])
    print("loaded trained model from", CKPT)
else:
    print("no checkpoint found — training a quick model (this is the Phase-4 recipe)")
    tokenizer = train_joint_bpe(data["train"], vocab_size=10000)
    V = tokenizer.get_vocab_size()
    cfg = ModelConfig.smoke(src_vocab_size=V, tgt_vocab_size=V)
    model = Transformer(cfg).to(device)
    loader = make_dataloader(data["train"][:4000], tokenizer, batch_size=32, shuffle=True, max_len=40)
    fit(model, loader, TrainConfig(warmup_steps=400, log_every=100), device, max_steps=400)
model.eval()
VOCAB = cfg.tgt_vocab_size
print("vocab:", VOCAB, "| params:", model.num_parameters())

## A reusable "translate" helper

Tokenize the English source the same way training did (tokens + `<eos>`), decode, and
strip special tokens for display.

In [ ]:
def encode_src(text, max_len=40):
    b = collate_fn([TranslationDataset([(text, "")], tokenizer, max_len)[0]])
    return b["src"].to(device), b["src_pad"].to(device)

def clean(ids):
    ids = [i for i in ids if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return tokenizer.decode(ids)

def translate_greedy(text, max_len=40):
    src, src_pad = encode_src(text, max_len)
    out = greedy_decode(model, src, src_pad, BOS_ID, EOS_ID, PAD_ID, max_len)
    return clean(out[0].tolist())

def translate_beam(text, beam_size=4, alpha=0.6, max_len=40):
    src, src_pad = encode_src(text, max_len)
    ids = beam_search(model, src, src_pad, BOS_ID, EOS_ID, PAD_ID, beam_size, alpha, max_len)
    return clean(ids)

## Feel-check: greedy-translate 10 examples and read them

After a full A100 run these should be fluent. After our tiny smoke run they'll be rough
but should be recognizably German with sensible structure. Read them — translation
quality is a thing you judge by eye, not only by BLEU.

In [ ]:
for en, de in data["validation"][:10]:
    print("EN  :", en)
    print("ref :", de)
    print("pred:", translate_greedy(en))
    print()

## Feel-check: greedy vs beam, side by side

Beam search explores more of the search space. Look for cases where greedy commits early
to a bad token and beam recovers, or where beam picks a more fluent phrasing.

In [ ]:
for en, de in data["validation"][:6]:
    print("EN    :", en)
    print("ref   :", de)
    print("greedy:", translate_greedy(en))
    print("beam=4:", translate_beam(en, beam_size=4))
    print()

## Feel-check: cross-attention alignment during generation

We greedy-decode one sentence and, at each generation step, record the **last decoder
layer's cross-attention** (averaged over heads) from the newest target position back to
every source token. Stacked, this is a (generated × source) alignment matrix. A
well-trained model produces a roughly monotonic diagonal-ish alignment (En→De reorders
some, so it won't be perfectly diagonal).

In [ ]:
@torch.no_grad()
def decode_with_alignment(text, max_len=40):
    src, src_pad = encode_src(text, max_len)
    layer = model.decoder.layers[-1].cross_attn
    layer.cache_attn = True
    memory = model.encode(src, src_pad)
    ys = torch.tensor([[BOS_ID]], device=device)
    rows = []
    for _ in range(max_len - 1):
        decoded = model.decode(ys, memory, ys.eq(PAD_ID), src_pad)
        rows.append(layer.last_attn[0, :, -1, :].mean(0).cpu())  # mean over heads -> (S,)
        nxt = model.output_proj(decoded[:, -1]).argmax(-1)
        ys = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
        if nxt.item() == EOS_ID:
            break
    layer.cache_attn = False
    align = torch.stack(rows)  # (T_gen, S)
    src_toks = [tokenizer.id_to_token(i) for i in src[0].tolist()]
    gen_toks = [tokenizer.id_to_token(i) for i in ys[0, 1:].tolist()]
    return align, src_toks, gen_toks

en = data["validation"][0][0]
align, src_toks, gen_toks = decode_with_alignment(en)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(align, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(src_toks))); ax.set_xticklabels(src_toks, rotation=90, fontsize=6)
ax.set_yticks(range(len(gen_toks))); ax.set_yticklabels(gen_toks, fontsize=6)
ax.set_xlabel("source (EN)"); ax.set_ylabel("generated (DE)")
ax.set_title(f"cross-attention alignment\nEN: {en}")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()
print("each generated row puts weight on the source tokens it 'looked at'")

## Takeaways

- Greedy is one path through the distribution; beam keeps several and length-normalizes.
- Cross-attention is interpretable: it shows the soft source↔target alignment the model
  learned, recovered for free from a model never trained on alignments.
- These decodes are rough only because the model is tiny and barely trained — the code
  is the same one you'll point at the A100 checkpoint.

**Next:** `06_evaluation.ipynb` — compute corpus **BLEU** with `sacrebleu` over the test
set, and report model size / steps / wall-clock / BLEU. Then we lift `fit()` into the
A100 training script and do the real run.